# CNN Image Classification (MNIST Sample)

## الترتيب / Flow
1. استيراد المكتبات - Import libraries
2. قراءة البيانات - Load dataset
3. تجهيز X و y - Reshape to (28, 28, 1) and normalize
4. تقسيم البيانات - Train/Test split
5. بناء CNN - Conv2D + MaxPooling + Dense
6. compile + fit - sparse categorical cross-entropy
7. التقييم - Test accuracy
8. عرض صور + تنبؤات - Visualize predictions

In [ ]:
# Step 1) استيراد المكتبات / Import libraries
# pip install tensorflow -q  # uncomment in Colab if needed
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

In [ ]:
# Step 2) قراءة البيانات / Load dataset
dataset = pd.read_csv('mnist_sample.csv')
print('Shape:', dataset.shape)
dataset.head()

In [ ]:
# Step 3) تجهيز X و y / Prepare and reshape images
y = dataset['label'].values.astype(int)
pixel_cols = [c for c in dataset.columns if c.startswith('pixel_')]
X = dataset[pixel_cols].values.reshape(-1, 28, 28, 1).astype('float32') / 255.0
print('X shape:', X.shape)
print('y shape:', y.shape)

In [ ]:
# Step 4) تقسيم البيانات / Train-Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y
)

In [ ]:
# Step 5) بناء CNN / Build model
model = Sequential([
    Conv2D(filters=32, kernel_size=(3, 3), activation='relu', input_shape=(28, 28, 1)),
    MaxPooling2D(pool_size=(2, 2)),
    Flatten(),
    Dense(units=128, activation='relu'),
    Dense(units=10, activation='softmax')
])
model.summary()

In [ ]:
# Step 6) compile + fit / Train model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
history = model.fit(
    X_train, y_train,
    batch_size=32,
    epochs=15,
    validation_split=0.2,
    verbose=1
)

In [ ]:
# Step 7) التقييم / Evaluate on test set
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f'Test loss: {loss:.4f}')
print(f'Test accuracy: {accuracy:.2%}')

In [ ]:
# Step 8) عرض صور + تنبؤات / Visualize predictions
y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)

fig, axes = plt.subplots(1, 5, figsize=(12, 3))
for i, ax in enumerate(axes):
    ax.imshow(X_test[i].reshape(28, 28), cmap='gray')
    color = 'green' if y_pred[i] == y_test[i] else 'red'
    ax.set_title(f'True: {y_test[i]} | Pred: {y_pred[i]}', color=color)
    ax.axis('off')
plt.suptitle('Sample Predictions (green=correct, red=wrong)')
plt.tight_layout()
plt.show()